# Indic Shloka → English translator (Colab)

**Workflow:** run the cells top to bottom → choose a model in the dropdown (cell 6) → click **Run** →
collect the Excel workbook from `Output_Folder/` (also offered as a download in cell 7).

Every record ends up in the output as `Success`, `Skipped` (unsupported language / empty text) or `Failed`
(with the reason in `Remarks`). Runs are checkpointed, so if Colab disconnects just run the cells again
with *Resume from checkpoint* ticked — finished records are not re-billed.

## 1 · Check the GPU
`Runtime → Change runtime type → Hardware accelerator: GPU`.

| Models | GPU needed |
|---|---|
| OpenAI (gpt-…) | none (runs on OpenAI servers) — a T4 is plenty for the local helpers |
| IndicTrans2, custom-shloka-mt | T4 (16 GB) is fine |
| Ollama 27–32B (qwen3:32b, qwen3:30b, gemma3:27b, aya-expanse:32b) | **L4 (24 GB) or A100** recommended — on a T4 layers spill to CPU and it is very slow |

In [ ]:
!nvidia-smi || echo "⚠ No GPU: switch the runtime to GPU (OpenAI models will still work; local models will be blocked)."

## 2 · Locate the repository (no hard-coded paths)
The cell finds the repository by its contents (`shloka_translator/__init__.py` + `config/config.yaml`),
not by a fixed path or folder name: first the current folder, then — in Colab — your Google Drive
(mounted on demand; an existing mount is reused), searched breadth-first. Optionally paste a folder into
`REPO_DIR_OVERRIDE` (or set the `SHLOKA_REPO_DIR` environment variable) to skip the search.

In [ ]:
# Locate the repository WITHOUT hard-coded paths:
#   1. REPO_DIR_OVERRIDE below or the SHLOKA_REPO_DIR environment variable (optional),
#   2. the current folder and its parents (works locally / in Jupyter),
#   3. Colab: mount Google Drive (re-using an existing mount) and search it for the folder that
#      contains shloka_translator/__init__.py + config/config.yaml — whatever its name or location.
import os
import sys
from pathlib import Path

REPO_DIR_OVERRIDE = ""  #@param {type:"string"}
MAX_SEARCH_DEPTH = 6    #@param {type:"integer"}
IN_COLAB = "google.colab" in sys.modules


def is_repo(path: Path) -> bool:
    return (path / "shloka_translator" / "__init__.py").is_file() and (path / "config" / "config.yaml").is_file()


def existing_drive_mount() -> Path | None:
    """Return the mount point of an already-mounted Google Drive, if any."""
    import psutil
    for part in psutil.disk_partitions(all=True):
        mp = Path(part.mountpoint)
        if mp.name == "drive" and ("fuse" in part.fstype.lower() or "drive" in part.device.lower()):
            return mp
    return None


def search(root: Path, max_depth: int) -> list[Path]:
    """Breadth-first, depth-limited search for repository folders under ``root``."""
    hits: list[Path] = []
    for depth in range(0, max_depth + 1):
        pattern = "/".join(["*"] * depth + ["shloka_translator", "__init__.py"])
        hits = [p.parent.parent for p in root.glob(pattern) if ".Trash" not in p.parts and is_repo(p.parent.parent)]
        if hits:
            break
    return hits


REPO_DIR = None
override = REPO_DIR_OVERRIDE.strip() or os.environ.get("SHLOKA_REPO_DIR", "").strip()
if override:
    REPO_DIR = Path(override).expanduser().resolve()
    if not is_repo(REPO_DIR):
        raise FileNotFoundError(f"{REPO_DIR} does not contain shloka_translator/ and config/config.yaml")
else:
    REPO_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if is_repo(p)), None)

if REPO_DIR is None and IN_COLAB:
    from google.colab import drive
    mount_point = existing_drive_mount()
    if mount_point is None:
        mount_point = Path.cwd() / "drive"      # Colab's default working folder + "drive"
        drive.mount(str(mount_point))
    print(f"Searching {mount_point} for the repository (depth ≤ {MAX_SEARCH_DEPTH})…")
    candidates = search(mount_point, MAX_SEARCH_DEPTH)
    if not candidates:
        raise FileNotFoundError("Repository not found on Google Drive. Upload the unzipped folder anywhere in "
                                "your Drive (or set REPO_DIR_OVERRIDE) and re-run this cell.")
    if len(candidates) > 1:
        print("Several copies found (using the most recently modified):", *candidates, sep="\n  ")
    REPO_DIR = max(candidates, key=lambda p: (p / "config" / "config.yaml").stat().st_mtime)

if REPO_DIR is None:
    raise FileNotFoundError("Repository not found. Start Jupyter inside the repo or set REPO_DIR_OVERRIDE.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("REPO_DIR =", REPO_DIR)

In [ ]:
import os, subprocess
os.chdir(REPO_DIR)

GITHUB_USER = "M2AI2015-Shounak"          # ← change
REPO_NAME   = "indic-shloka-translator"       # ← name from Step 1
GIT_NAME    = "Shounak Sushanta Dasgupta"
GIT_EMAIL   = "m25ai2015@iitj.ac.in"               # ← the email on your GitHub account

def git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    return r.returncode

git("init", "-b", "main")
git("config", "user.name", GIT_NAME)
git("config", "user.email", GIT_EMAIL)

In [ ]:
# 1) Clear notebook outputs (they can contain run logs, paths or file previews)
!pip -q install nbstripout
!nbstripout notebooks/*.ipynb

# 2) Check that nothing secret or bulky gets committed
git("add", "-A")
git("status", "--short")

# 3) Look for key-like strings in what is about to be committed
!git grep -nE "sk-[A-Za-z0-9_-]{20,}|hf_[A-Za-z0-9]{20,}|ghp_|github_pat_" -- . ':!tests' ':!README.md' || echo "✓ no secrets found"

# 4) GitHub rejects any single file over 100 MB
!find . -path ./.git -prune -o -type f -size +50M -print

## 3 · Install dependencies (fault-tolerant)
Installs `requirements.txt` (torch is **not** reinstalled — Colab's CUDA build is used).
A plain `pip install -r` installs **nothing** if a single package fails to build, which makes every
local model show *"… package not installed"*. `scripts/install_requirements.py` therefore falls back to
installing package by package and prints a ✓/✗ table, so one failure only disables the feature that
needs it. The second part prints which model families are usable and why.

It also downloads AI4Bharat **IndicLID** (if `language_detection.use_indiclid` is on; cached under
`Output_Folder/.cache/IndicLID` so it is fetched once).

> If the report says a package that was already imported was upgraded (e.g. `transformers`), use
> **Runtime → Restart session** and run the notebook again from cell 2.

In [ ]:
!python scripts/install_requirements.py requirements.txt

import importlib
import importlib.metadata as md
importlib.invalidate_caches()

CHECKS = [  # (module to import, pip package, what needs it)
    ("openai", "openai", "OpenAI models"),
    ("ollama", "ollama", "Ollama models (Python client)"),
    ("transformers", "transformers", "IndicTrans2 / custom model"),
    ("IndicTransToolkit", "IndicTransToolkit", "IndicTrans2 / custom model (pre/post-processing)"),
    ("torch", "torch", "IndicTrans2 / custom model (Colab preinstalled)"),
    ("pynvml", "nvidia-ml-py", "GPU_Metrics sheet"),
    ("fasttext", "fasttext-wheel", "IndicLID + fastText language-ID (optional)"),
    ("sanskrit_parser", "sanskrit_parser", "rule-based sandhi split (optional)"),
]
for module, pkg, purpose in CHECKS:
    try:
        importlib.import_module(module)
        try:
            version = md.version(pkg)
        except md.PackageNotFoundError:
            version = "?"
        print(f"✓ {pkg:18s} {version:12s} {purpose}")
    except Exception as exc:  # noqa: BLE001
        print(f"✗ {pkg:18s} {'missing':12s} {purpose}  →  !pip install {pkg}   ({type(exc).__name__})")

from shloka_translator.config import load_config
from shloka_translator.langid.detector import install_indiclid
cfg = load_config()
if cfg.language_detection.use_indiclid:
    ok = install_indiclid(cfg.path(cfg.language_detection.indiclid_dir))
    print("IndicLID ready" if ok else "IndicLID unavailable — rule-based + fastText detection will be used")

## 4 · Secrets (never typed into the notebook)
Click the **🔑 key icon** in the left sidebar → **Add new secret**:

* `OPENAI_API_KEY` — needed for OpenAI models, the analysis helper and the language-ID tiebreaker
* `HF_TOKEN` — needed for IndicTrans2 / BPCC (first accept the licence on
  https://huggingface.co/ai4bharat/indictrans2-indic-en-1B)

and switch **Notebook access** on for both. Values are never printed or logged.

In [ ]:
from shloka_translator.secrets import get_secret, install_redaction
import logging
install_redaction(logging.getLogger())
OPENAI_API_KEY = get_secret("OPENAI_API_KEY")
HF_TOKEN = get_secret("HF_TOKEN")
print("OPENAI_API_KEY:", "set" if OPENAI_API_KEY else "missing")
print("HF_TOKEN:      ", "set" if HF_TOKEN else "missing")
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)

## 5 · Ollama setup (only for Ollama models)
Ollama models are **not** a hosted API: they are downloaded and run **inside this Colab VM**, and the code
talks to the local server at `http://localhost:11434`.
List the model(s) you plan to use below, or leave the list empty — the **Run** button (and the
*Set up Ollama* button) in cell 6 installs Ollama and pulls the selected model automatically.

Optional: set `OLLAMA_MODELS` to a Drive folder (the cell suggests one next to the repository) so downloads survive runtime resets
(≈75 GB for all four models; loading from Drive is slower).

In [ ]:
OLLAMA_SETUP_MODELS = []          # e.g. ["qwen3:32b"]  or  ["--all"]
# Optional persistent model store next to the repo on Drive (~75 GB for all four models):
# os.environ["OLLAMA_MODELS"] = str(REPO_DIR.parent / "ollama-models")

if OLLAMA_SETUP_MODELS:
    os.environ["OLLAMA_NUM_PARALLEL"] = str(cfg.concurrency.ollama_max_workers)
    !bash scripts/setup_ollama.sh {" ".join(OLLAMA_SETUP_MODELS)}
else:
    print("Skipped (no Ollama model listed). The Run button sets Ollama up on demand.")

## 6 · Choose a model and run
The dropdown is grouped by provider and shows availability: **✓ ready**, **⬇ set up automatically when you click Run** (Ollama server + model download, missing `ollama` / `IndicTransToolkit` packages), or **⚠ reason** (needs an action from you)
(e.g. *not available for this key*, *not pulled yet*, *Custom model not trained yet*).
`★ ALL MODELS` runs every model one after another and also writes a side-by-side comparison workbook.

* **Concurrency** — parallel requests for OpenAI / Ollama (IndicTrans2 & custom always batch on the GPU).
* **Max records** — `0` = everything; e.g. `5` for a smoke test.
* **Resume from checkpoint** — reuse finished records after a disconnect (no re-billing).

**What the ⚠ messages mean and how to fix them**

| Message | Meaning | Fix |
|---|---|---|
| ⬇ `not set up yet — Run installs Ollama…` | normal on a fresh runtime: Ollama models are not a hosted API — the `ollama` client, the Ollama server and the model weights (17–20 GB each) are installed **inside this Colab VM** | just click **Run** (or **Set up Ollama for selection** first); use an L4/A100 runtime for 27–32B models |
| `Ollama server not running` | normal before setup — the models are not downloaded yet | select the model and click **Set up Ollama for selection** (or just **Run**, which does it automatically); needs an L4/A100 for 27–32B models |
| `not pulled yet` | server is running, model not downloaded | same as above |
| ⬇ `IndicTransToolkit missing — Run installs it` | the IndicTrans2 helper package failed to install in cell 3 | click **Run** (it retries PyPI, then the GitHub source); if it still fails, read the ✗ line printed by cell 3 |
| `transformers not installed` | the core install in cell 3 failed | re-run cell 3 and read its ✗ lines, then **Refresh availability** |
| `HF_TOKEN not set` / `No access …` | IndicTrans2 is a gated model | accept the licence on its Hugging Face page, add `HF_TOKEN` in 🔑 Secrets, re-run cell 4 |
| `No GPU …` | CPU runtime | Runtime → Change runtime type → GPU |
| `not available for this key` | your OpenAI key cannot use that model name | pick another OpenAI model |
| `Custom model not trained yet` | no checkpoint in `models/custom/` | run `02_train_custom_model.ipynb` |

In [ ]:
from shloka_translator.ui.widgets import build_panel
panel = build_panel()

## 7 · Results
Shows the first rows of `Translations`, status counts, token totals and the output path, and offers the
workbook(s) as a download. (Run this after the run in cell 6 has finished.)

In [ ]:
import pandas as pd
from shloka_translator.ui.widgets import latest_result

res = latest_result()
if res is None or not res.ok:
    print("No finished run yet — choose a model in cell 6 and click Run, then re-run this cell."
          if res is None else res.message)
else:
    for m in res.models:
        print(f"== {m.model}: {m.counts} | tokens {m.tokens}\n   {m.output_path}")
        display(pd.read_excel(m.output_path, sheet_name="Translations", dtype=str).head(10))
    try:
        from google.colab import files
        for m in res.models:
            files.download(str(m.output_path))
        if res.comparison_path:
            files.download(str(res.comparison_path))
    except Exception as exc:
        print("Download not offered:", exc, "— the files are in Output_Folder/ on your Drive.")

## 8 · Optional: run ALL models sequentially (no widget)
Same as choosing `★ ALL MODELS` in the dropdown. Models that are not available are still listed in their
workbook with `Failed` records and the reason. Ollama models are set up first.

In [ ]:
RUN_ALL = False            # set True to run
if RUN_ALL:
    from shloka_translator.pipeline.orchestrator import RunOptions, run_pipeline
    !bash scripts/setup_ollama.sh --all
    all_res = run_pipeline(load_config(), RunOptions(model="ALL MODELS", max_records=None,
                                                     openai_key=OPENAI_API_KEY, hf_token=HF_TOKEN))
    print(all_res.message, "→", all_res.comparison_path)
    if all_res.comparison_path:
        display(pd.read_excel(all_res.comparison_path, sheet_name="Comparison").head(10))